# Uuringu "Tarbijate hoiakud tekstiilide liigiti kogumisel rõivastest ja kodutekstiilidest loobumisel" vastuste analüüs

## Vajalike pakettide impordid ja seadistused

In [125]:
from typing import Callable

import gspread
import pandas as pd
import duckdb
import re

In [215]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 45)

In [ ]:
def _register_duckdb_function(duckdb_function_name: str, function: Callable) -> None:
    try:
        duckdb.remove_function(duckdb_function_name)
    except:
        pass
    duckdb.create_function(duckdb_function_name, function)

## Andmete laadimine

In [338]:
# Autentimine
gc = gspread.service_account(filename="freanalyys-732d478588e9.json", scopes=gspread.auth.READONLY_SCOPES)

# Leia ja ava Google Sheets dokument ning selles vajalik tööleht
sh = gc.open("Uuring Tarbijate hoiakud tekstiilide liigiti kogumisel rõivastest ja kodutekstiilidest loobumisel (vastused)")

# Leia vastuste tööleht
vastused = sh.worksheet("Vormi vastused")
# Päri kõik vastused
data_raw  = pd.DataFrame(vastused.get_all_records(default_blank="NULL"))

# Leia vastuste koodude tööleht
v_koodid = sh.worksheet("Vastuste koodid")
# Päri kõik vastuste koodid
vastuste_koodid  = pd.DataFrame(v_koodid.get_all_records())

## Andmetega tutvumine ja andmete puhastamine

In [238]:
duckdb.sql("""SUMMARIZE data_raw""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Q1_aeg,VARCHAR,11/14/2024 15:36:53,5/8/2025 22:30:45,869,NaN,NaN,NaN,NaN,NaN,678,0.0
1,Q2_kontroll,BIGINT,1,2,2,1.0073746312684366,0.08562160272537057,1,1,1,678,0.0
2,Q3_vanus,BIGINT,1,5,5,2.8436578171091447,0.8640426850655679,2,3,3,678,0.0
3,Q4_sugu,BIGINT,1,3,3,1.9056047197640118,0.33496114562237966,2,2,2,678,0.0
4,Q5_elukoht,VARCHAR,Ameerika Ühendriigid,"Ülenurme, Kambja vald",157,NaN,NaN,NaN,NaN,NaN,678,0.0
5,Q6_keel,BIGINT,1,4,4,1.0870206489675516,0.42774145221414595,1,1,1,678,0.0
6,Q6_keel_tekst,VARCHAR,Eesti ja vene keel,eesti + hollandi + inglise,7,NaN,NaN,NaN,NaN,NaN,678,0.0
7,Q7_sorteerimiskaitumine,VARCHAR,Ei sorteeri,ühistu mittetoimivuse tõttu on teema raskendatud,11,NaN,NaN,NaN,NaN,NaN,678,0.0
8,Q8_teadlikkus,BIGINT,1,5,5,3.4690265486725664,0.915038930589041,3,4,4,678,0.0
9,Q9_probleemi_tosidus,BIGINT,1,5,5,4.314159292035399,0.9524004274952562,4,5,5,678,0.0


### Q1_aeg

In [239]:
# Leia vastuste min ja max kuupäevad. Kas kattuvad uuringu perioodiga?
duckdb.sql('''
    SELECT
        MIN(strptime(Q1_aeg, '%m/%d/%Y %H:%M:%S')) as min,
        MAX(strptime(Q1_aeg, '%m/%d/%Y %H:%M:%S')) as max
    FROM data_raw
''').df()

,min,max
0,2024-11-09 17:46:31,2025-05-08 22:30:45


In [ ]:
# Kontrolli, kas esineb duplikaate, et välistada uuringu vastuste topelt esitamist, bot'ide tegevust
timestamp_duplicates = duckdb.sql('''
    SELECT
        strptime(K1_aeg, '%m/%d/%Y %H:%M:%S') as ajatempel,
        K3_vanus,
        K4_sugu,
        K5_elukoht,
        count(*) as count
    FROM data_raw
    GROUP BY strptime(K1_aeg, '%m/%d/%Y %H:%M:%S'), K4_sugu, K5_elukoht
    HAVING count(*) > 1
''')
timestamp_duplicates.df()

,ajatempel,K4_sugu,K5_elukoht,count
0,2025-03-13 11:29:17,Mees,Järvamaa,2


In [224]:
duckdb.sql('''
    SELECT *
    FROM data_raw
    WHERE strptime(Q1_aeg, '%m/%d/%Y %H:%M:%S') IN (
        '2025-03-20 11:47:56',
        '2025-03-20 11:46:06',
        '2025-03-13 10:32:53',
        '2025-03-13 11:29:17'
    )
    ORDER BY strptime(Q1_aeg, '%m/%d/%Y %H:%M:%S')
''').df()

,Q1_aeg,Q2_kontroll,Q3_vanus,Q4_sugu,Q5_elukoht,Q6_keel,Q7_sorteerimiskaitumine,Q8_teadlikkus,Q9_probleemi_tosidus,Q10_probleemi_tosidus_tapsustus,Q11_teadlikkus,Q12_teadlikkus_tapsustus,Q13_kommunikatsiooni_selgus,Q14_tekstiilide_kogus,Q15_mittevajalikud_tekstiilid,Q16_mittevajalikud_tekstiilid_tapsustus,Q17_valmisolek_parandamiseks,Q18_valmisolek_parandamiseks_tapsustus,Q19_loobumise_pohjused,Q20_loobumise_pohjused_tapsustus,Q21_loobumise_lihtsus,Q22_peamised_väljakutsed,Q23_kasutuskolbmatud_tekstiilid,Q24_kasutuskolbmatud_tekstiilid_tapsustus,Q25_loobumisel_oluline,Q26_korduskasutuseks_sobimatud_tekstiilid,Q27_korduskasutuseks_sobimatud_tekstiilid_tapsustus,Q28_riikliku_juhise_selgus,Q29_sobiv_kogumisviis,Q30_valmisolek_kategoriseerimiseks,Q31_valmisolek_kategoriseerimiseks,Q32_teabe_allikad,Q33_julgustavad_tegurid,Q34_ettepanekud,Q35_valmisolek_telefonikoneks,Q36_valmisolek_lisakysimusteks,Q37_rahulolu_garderoobiga,Q38_kandmise_kestus,Q39_kandmise_kestus_tapsustus,Q40_ostmissagedus,Q41_ultrakiirmoe_ostmine,Q42_ultrakiirmoe_eelistamise_pohjused,Q43_roivaste_tellimine_proovimiseks
0,3/13/2025 10:32:53,Jah,Alaealine (17 või noorem),Mees,Järvamaa,Eesti keel,Ei tea,3,4,NULL,Ei ole teadlik,NULL,Puudulik,Ei tea,Ei tea,NULL,2,NULL,"Katkised või määrdunud, Vale soetatud suurus",NULL,4,Ma ei sorteeri neid,.,NULL,-,Ei,NULL,Ei ole sellega kursis/ei tea,Kogumisnädalad kodukandis (nt kord kvartalis),"Jah, kui kogumispunktid on mugavas asukohas ja...",NULL,"See teema ei ole mind kunagi paelunud, ei huvi...",Kogumispunktide lähedus elukohale ja ligipääse...,Ei ole,Ei,Ei,NULL,NULL,NULL,NULL,NULL,NULL,NULL
1,3/13/2025 10:32:53,Jah,Alaealine (17 või noorem),Naine,Järvamaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",2,3,NULL,Ei ole teadlik,NULL,Arusaamatu,Kuni 1kg,"Müün teisel ringil (veebis), Müün teisel ringi...",NULL,3,NULL,"Trend/mood on aegunud, Katkised või määrdunud,...",NULL,3,Ajapuudus,"Ei tea, sest mina ei vastuta selle tegevuse ee...",NULL,Mind ei huvita eseme edasine saatus,Jah,NULL,Osaliselt arusaadav,Kogumispunktid kaubanduspindadel,"Jah, kui sorteerimisjuhis on selge ja konkreetne",NULL,"Televisioon, Ajalehed, ajakirjad, Sotsiaalmeedia",Soodustused/allahindlused kauplustes,NULL,ei,Ei,NULL,NULL,NULL,NULL,NULL,NULL,NULL
2,3/13/2025 11:29:17,Jah,Alaealine (17 või noorem),Mees,Järvamaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,4,NULL,Ei ole teadlik,NULL,Puudulik,Kuni 10kg,"Müün teisel ringil (füüsilises poes), Annan pe...",NULL,3,NULL,"Katkised või määrdunud, Kulunud või ilme kaotanud",NULL,4,ei oska vastata,Viin eraldi jäätmejaama,NULL,"Heategevuslik aspekt, Rõivale on tagatud uus elu",Ei,NULL,Osaliselt arusaadav,ei tea,Pole kindel,NULL,"Televisioon, Sotsiaalmeedia, Sõpradelt, tuttav...",Selged juhised,ei oska öelda,ei,Ei,NULL,NULL,NULL,NULL,NULL,NULL,NULL
3,3/13/2025 11:29:17,Jah,18-29,Mees,Järvamaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",5,4,Eestis pole nii hull olukord,"Jah, olen teadlik",NULL,"Selge, kuid mittetäielik",Kuni 5kg,"Müün teisel ringil (veebis), Annan perele või ...",NULL,3,NULL,"Trend/mood on aegunud, Katkised või määrdunud,...",NULL,5,Tekstiilide transport kogumispunktidesse on ke...,"Ei tea, sest mina ei vastuta selle tegevuse ee...",NULL,"Heategevuslik aspekt, Rõivale on tagatud uus elu",Ei,NULL,Täiesti arusaadav,Püsivad kogumispunktid kohalikus jäätmejaamas,"Jah, see oleks mulle sobiv",NULL,"Artiklid veebiväljaannetes, Sotsiaalmeedia, Sõ...",Kogumispunktide lähedus elukohale ja ligipääse...,NULL,Ei,Jah,5,"Mu garderoobis on pikaealised riided, mida hoo...",Pusa ja püksid kestavad aastaid,Kord kuus,3,Ilusad riided kiirelt,3
4,3/20/2025 11:46:06,Jah,18-29,Naine,"Harjumaa, Lääne-Harju vald",Vene keel,"Jah, 3-5 jäätmekategoorias",4,3,NULL,"Olen kuulnud, kuid täpsemalt ei tea",NULL,"Selge, kuid mittetäielik",Kuni 10kg,"Müün teisel ringil (veebis), Annan perele või ...",NULL,4,NULL,"Trend/mood on aegunud, Kulunud või ilme kaotan...",NULL,4,Tekstiilide transport kogumispunktidesse on ke...,Rõivakonteineritesse,NULL,Rõivas

### Q2_kontroll

In [295]:
# Leia read, kus kontrollküsimuse vastus = 'Ei'. Need read vastustest eemaldada.
duckdb.sql('''
    SELECT count(*) as vastuste_arv
    FROM data_raw t1
    JOIN vastuste_koodid t2 ON t1.Q2_kontroll = t2.kood
        AND t2.kysimus = 'Q2_kontroll'
    WHERE t2.vastus = 'Ei'
''').df()

,vastuste_arv
0,5


### Q3_vanus

In [318]:
# Leia unikaalsed vastusevariandid. Kas esineb 5 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q3_vanus,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 on t1.Q3_vanus = t2.kood
        AND t2.kysimus = 'Q3_vanus'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q3_vanus, t2.vastus
    ORDER BY t1.Q3_vanus
''').df()

,Q3_vanus,Vastus,vastuste_arv
0,1,Alaealine (17 või noorem),57
1,2,18-29,124
2,3,30-49,370
3,4,50-64,104
4,5,65 või vanem,13


### Q4_sugu

In [319]:
# Leia unikaalsed vastusevariandid. Kas esineb 3 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q4_sugu,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q4_sugu = t2.kood
        AND t2.kysimus = 'Q4_sugu'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q4_sugu, t2.vastus
    ORDER BY t1.Q4_sugu
''').df()

,Q4_sugu,Vastus,vastuste_arv
0,1,Mees,71
1,2,Naine,591
2,3,Ei soovi määratleda,6


### Q5_elukoht

In [320]:
# Leia unikaalsed vastusevariandid
duckdb.sql('''
    SELECT
        DISTINCT Q5_elukoht,
        count(*) as vastuste_arv
    FROM data_raw
    WHERE
        strptime(Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY Q5_elukoht
''').df()

,Q5_elukoht,vastuste_arv
0,"Keila, Harjumaa",1
1,Läänemaa,1
2,Viljandi mk,1
3,Rummu,1
4,Järva-Jaani Järvamaa,2
...,...,...
138,"Harju maakond, Saue vald",1
139,Tartu maakond,2
140,Viimsi,1
141,Järvamaa,15


In [334]:
def paranda_elukoha_vead(elukoht: str) -> str:
    '''
    Tagasta teadaolevate kirjavigadega elukoha nimetuse asemel korrektne nimetus, kust on elukoha eest ning järelt eemaldatud üleliigsed tühikud.
    '''
    elukoht = elukoht.lower().strip()
    
    # Paranda enamlevinud kirjavead
    asendused = {
        "tln": "tallinn",
        "tallinm": "tallinn",
        "harjumas": "harjumaa",
        "hafjumas": "harjumaa",
        "hatjumaa": "harjumaa",
        "pärjumaa": "pärnumaa",
        "jõheva": "jõgeva",
        "l-viru": "lääne-virumaa",
        "ida-virimaa": "ida-virumaa",
        "jarva maa": "järvamaa",
        "parnu": "pärnu",
        "järva jaani": "järva-jaani"
    }
    
    for k, v in asendused.items():
        elukoht = elukoht.replace(k, v)
    
    return elukoht

# Teadaolevate maakonna kirjapiltide vastavuspaarid
county_map = {
    "harjumaa": "Harju maakond",
    "harju maakond": "Harju maakond",
    "harju mk": "Harju maakond",
    "harju": "Harju maakond",
    "hiiumaa": "Hiiu maakond",
    "hiiu maakond": "Hiiu maakond",
    "ida-virumaa": "Ida-Viru maakond",
    "ida-viru maakond": "Ida-Viru maakond",
    "ida-viru": "Ida-Viru maakond",
    "jõgevamaa": "Jõgeva maakond",
    "jõgeva maakond": "Jõgeva maakond",
    "järvamaa": "Järva maakond",
    "järva maakond": "Järva maakond",
    "läänemaa": "Lääne maakond",
    "lääne maakond": "Lääne maakond",
    "lääne-virumaa": "Lääne-Viru maakond",
    "lääne-viru maakond": "Lääne-Viru maakond",
    "lääne-viru": "Lääne-Viru maakond",
    "põlvamaa": "Põlva maakond",
    "põlva maakond": "Põlva maakond",
    "pärnumaa": "Pärnu maakond",
    "pärnu maakond": "Pärnu maakond",
    "raplamaa": "Rapla maakond",
    "rapla maakond": "Rapla maakond",
    "saaremaa": "Saare maakond",
    "saare maakond": "Saare maakond",
    "tartumaa": "Tartu maakond",
    "tartu maakond": "Tartu maakond",
    "valgamaa": "Valga maakond",
    "valga maakond": "Valga maakond",
    "viljandimaa": "Viljandi maakond",
    "viljandi maakond": "Viljandi maakond",
    "võrumaa": "Võru maakond",
    "võru maakond": "Võru maakond"
}

# Linna ja maakonna vastavuspaarid
place_to_county = {
    # Harjumaa
    "tallinn": "Harju maakond",
    "keila": "Harju maakond",
    "maardu": "Harju maakond",
    "paldiski": "Harju maakond",
    "viimsi": "Harju maakond",
    "saku": "Harju maakond",
    "saue": "Harju maakond",
    "rae": "Harju maakond",
    "harku": "Harju maakond",
    "lääne-harju": "Harju maakond",
    "klooga": "Harju maakond",
    "rummu": "Harju maakond",
    "padise": "Harju maakond",
    # Ida-Virumaa
    "jõhvi": "Ida-Viru maakond",
    # Jõgevamaa
    "jõgeva": "Jõgeva maakond",
    "luua": "Jõgeva maakond",
    "põltsamaa": "Jõgeva maakond",
    # Järvamaa
    "järva-jaani": "Järva maakond",
    "järva vald": "Järva maakond",
    # Lääne-Virumaa
    "rakvere": "Lääne-Viru maakond",
    # Põlvamaa
    "põlva": "Põlva maakond",
    # Pärnumaa
    "pärnu": "Pärnu maakond",
    "sindi": "Pärnu maakond",
    # Raplamaa
    "rapla": "Rapla maakond",
    "raka": "Rapla maakond",
    # Tartumaa
    "tartu": "Tartu maakond",
    "ülenurme": "Tartu maakond",
    "kambja": "Tartu maakond",
    # Valgamaa
    "valga": "Valga maakond",
    # Viljandimaa
    "viljandi": "Viljandi maakond",
    "põhja-sakala": "Viljandi maakond"
}

def tuleta_maakond(tekst: str) -> str:
    '''
    Tagasta etteantud sõnastike alusel vastaja elukoha maakond
    '''
    elukoht = paranda_elukoha_vead(tekst)
    elukohad = re.split(r"[,/]| ja ", elukoht)
    peamine_elukoht = elukohad[0].strip()

    # 1. direct county match
    for i in county_map:
        if i in peamine_elukoht:
            return county_map[i]

    # 2. place → county
    for i in place_to_county:
        if i in peamine_elukoht:
            return place_to_county[i]

    return "Muu"

# Registreeri funktsioon duckdb-ga
_register_duckdb_function("tuleta_maakond", tuleta_maakond)

data_raw["K5_elukoht"].apply(tuleta_maakond)

0      Harju maakond
1      Pärnu maakond
2      Harju maakond
3                Muu
4      Tartu maakond
           ...      
673    Pärnu maakond
674    Harju maakond
675    Harju maakond
676    Harju maakond
677    Pärnu maakond
Name: K5_elukoht, Length: 678, dtype: str

In [335]:
duckdb.sql('''
    SELECT
        --,Q5_elukoht
        tuleta_maakond(K5_elukoht) as maakond
        ,t2.kood
        ,count(*)
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 on t1.K5_elukoht = t2.Vastus
        AND t2.kysimus = 'K5_elukoht'
    group by tuleta_maakond(K5_elukoht),t2.kood
    order by t2.kood
    
''').df()

,maakond,Kood,count_star()
0,Harju maakond,1,11
1,Tartu maakond,12,3
2,Järva maakond,4,3
3,Pärnu maakond,9,1
4,Pärnu maakond,NaN,56
5,Rapla maakond,NaN,13
6,Võru maakond,NaN,4
7,Jõgeva maakond,NaN,7
8,Järva maakond,NaN,33
9,Ida-Viru maakond,NaN,3


### Q6_keel

In [321]:
# Leia unikaalsed vastusevariandid. Kas on 4 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q6_keel,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q6_keel = t2.kood
        AND t2.kysimus = 'Q6_keel'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q6_keel, t2.vastus
    ORDER BY t1.Q6_keel
''').df()

,Q6_keel,Vastus,vastuste_arv
0,1,Eesti keel,641
1,2,Inglise keel,7
2,3,Vene keel,15
3,4,Muu,5


In [212]:
def teisenda_keel(tekst: str) -> str:
    '''
    Tagasta keelevalikud: eesti, vene, inglise, muudel juhtudel "muu"
    '''
    keel = tekst.lower().split("keel")[0].strip()
    if keel in ("eesti", "vene", "inglise"):
        return keel
    return "Muu"

_register_duckdb_function("teisenda_keel", teisenda_keel)

### Q7_sorteerimiskaitumine

In [322]:
# Leia unikaalsed vastusevariandid. Kas on 6 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q7_sorteerimiskaitumine,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q7_sorteerimiskaitumine = t2.kood
        AND t2.kysimus = 'Q7_sorteerimiskaitumine'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q7_sorteerimiskaitumine, t2.vastus
    ORDER BY t1.Q7_sorteerimiskaitumine
''').df()

,Q7_sorteerimiskaitumine,Vastus,vastuste_arv
0,1,Ei sorteeri,30
1,2,"Jah, maksimaalselt kahes kategoorias",73
2,3,"Jah, 3-5 jäätmekategoorias",424
3,4,"Jah, üle 6 jäätmekategoorias",134
4,5,"Ei, sest elan täielikult null-kulu eluviisi",1
5,6,Muu,6


### Q8_teadmiste_hinnang

In [323]:
# Leia unikaalsed vastusevariandid. Kas on 5 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q8_teadmiste_hinnang,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q8_teadmiste_hinnang = t2.kood
        AND t2.kysimus = 'Q8_teadmiste_hinnang'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q8_teadmiste_hinnang, t2.vastus
    ORDER BY t1.Q8_teadmiste_hinnang
''').df()

,Q8_teadmiste_hinnang,Vastus,vastuste_arv
0,1,Väga madal või olematu tarbijateadlikkus,19
1,2,puudu,58
2,3,puudu,243
3,4,puudu,279
4,5,Kõik minu valikud on teistele eeskujuks,69


### Q9_probleemi_tosidus

In [324]:
# Leia unikaalsed vastusevariandid. Kas on 5 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q9_probleemi_tosidus,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q9_probleemi_tosidus = t2.kood
        AND t2.kysimus = 'Q9_probleemi_tosidus'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q9_probleemi_tosidus, t2.vastus
    ORDER BY t1.Q9_probleemi_tosidus
''').df()

,Q9_probleemi_tosidus,Vastus,vastuste_arv
0,1,Ei näe seda sugugi probleemina,9
1,2,puudu,23
2,3,puudu,98
3,4,puudu,152
4,5,"Äärmiselt suur probleem, millel on tõsised tag...",386


### Q11_teadlikkus

In [325]:
# Leia unikaalsed vastusevariandid. Kas on 4 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q11_teadlikkus,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q11_teadlikkus = t2.kood
        AND t2.kysimus = 'Q11_teadlikkus'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q11_teadlikkus, t2.vastus
    ORDER BY t1.Q11_teadlikkus
''').df()

,Q11_teadlikkus,Vastus,vastuste_arv
0,1,"Jah, olen teadlik",252
1,2,"Olen kuulnud, kuid täpsemalt ei tea",222
2,3,"Olen teadlik, aga see on valitsuse ja KOV'ide,...",5
3,4,Ei ole teadlik,189


### Q13_kommunikatsiooni_selgus

In [336]:
# Leia unikaalsed vastusevariandid. Kas on 4 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q13_kommunikatsiooni_selgus,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q13_kommunikatsiooni_selgus = t2.kood
        AND t2.kysimus = 'Q13_kommunikatsiooni_selgus'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q13_kommunikatsiooni_selgus, t2.vastus
    ORDER BY t1.Q13_kommunikatsiooni_selgus
''').df()

BinderException: Binder Error: Table "t1" does not have a column named "Q13_kommunikatsiooni_selgus"

Candidate bindings: : "K13_kommunikatsiooni_selgus"

### Q14_tekstiilide_kogus

In [ ]:
# Leia unikaalsed vastusevariandid. Kas on 7 kategooriat?
duckdb.sql('''
    SELECT DISTINCT
        t1.Q14_tekstiilide_kogus,
        t2.vastus,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 ON t1.Q14_tekstiilide_kogus = t2.kood
        AND t2.kysimus = 'Q14_tekstiilide_kogus'
    WHERE
        strptime(t1.Q1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND t1.Q2_kontroll = 1 --eemaldada read, kus kontrollküsimuse vastus on "ei"
    GROUP BY t1.Q14_tekstiilide_kogus, t2.vastus
    ORDER BY t1.Q14_tekstiilide_kogus
''').df()

,Q14_tekstiilide_kogus,Vastus,vastuste_arv
0,1,Kuni 1kg,109
1,2,Kuni 5kg,294
2,3,Kuni 10kg,155
3,4,Kuni 15kg,54
4,5,Üle 15kg,20
5,6,Ei oska öelda,22
6,7,Muu,14




### Group C — Behaviour (Multi-select strings)

These columns store comma-separated multi-choice answers as a single string.

| Col | Variable | Checks |
|---|---|---|
| **Q13** | What do you do with unwanted clothes | Multi-select string — split by comma/delimiter; verify each token matches expected option set; flag novel free-text in Q13.1 |
| **Q14** | Repair willingness (1–5) | Range 1–5 ✓; check for consistency with Q13 (if someone selects "repair" in Q13, Q14 score should not be 1) |
| **Q15** | Reasons for discarding | Multi-select string — same parsing check as Q13 |
| **Q16** | Ease of discarding (1–5) | Range 1–5 ✓ |
| **Q17** | Sorting challenges | Multi-select — parse and verify against expected options |
| **Q18** | Where do unusable textiles go | Multi-select — check for internal contradictions (e.g., respondent answers both "container" and "landfill bin") |
| **Q19** | What matters when discarding | Multi-select — parse and verify |
| **Q20** | Knowingly placed unsuitable items in container | 3 values — verify expected set (Jah/Ei/Olen kahelnud or similar) |
| **Q21** | National guideline clarity | **Critical issue:** Contains both text labels ("Osaliselt arusaadav") AND numeric codes (1, 2, 3, 4) — mixed encoding. Must unify to one scale before analysis. 3 NaNs also present. |
| **Q22** | Preferred collection method | 26 unique values — likely multi-select; parse and validate |
| **Q23** | Willingness to use multiple containers | 11 unique values — check for expected options |
| **Q24** | Information sources | Multi-select — parse and count co-occurrences |
| **Q25** | Motivators for sorting | Multi-select — parse and verify options |

### Puhastatud andmestik

In [343]:
# Puhastatud andmed
data = duckdb.sql('''
    SELECT
        strptime(K1_aeg, '%m/%d/%Y %H:%M:%S') as ajatempel
        ,K3_vanus as vanus
        ,K4_sugu as sugu
        --,Q5_elukoht
        ,tuleta_maakond(K5_elukoht) as maakond
        ,K6_keel
        --,teisenda_keel(K6_keel) as peamine_kodune_keel
        ,K7_sorteerimiskaitumine
    FROM data_raw
    WHERE
        strptime(K1_aeg, '%m/%d/%Y %H:%M:%S') BETWEEN '2025-03-10 00:00:00' AND '2025-04-30 23:59:59'
        AND K2_kontroll = 'Jah' --eemaldada read, kus kontrollküsimuse vastus on "ei"
        -- mis saab ridadest, kus Q13_kommunikatsiooni_selgus on tühi? kattuvad valede kuupäevadega
''').df()
data

,ajatempel,vanus,sugu,maakond,K6_keel,K7_sorteerimiskaitumine
0,2025-03-10 08:01:13,18-29,Naine,Harju maakond,Eesti keel,"Jah, maksimaalselt kahes kategoorias"
1,2025-03-10 16:55:18,18-29,Naine,Pärnu maakond,Eesti keel,"Jah, 3-5 jäätmekategoorias"
2,2025-03-17 21:06:37,50-64,Naine,Harju maakond,Eesti keel,"Jah, maksimaalselt kahes kategoorias"
3,2025-03-17 20:00:21,50-64,Naine,Muu,Eesti keel,"Jah, 3-5 jäätmekategoorias"
4,2025-03-20 21:15:13,18-29,Naine,Tartu maakond,Eesti keel,"Jah, 3-5 jäätmekategoorias"
...,...,...,...,...,...,...
663,2025-03-28 16:34:49,50-64,Naine,Pärnu maakond,Eesti keel,"Jah, maksimaalselt kahes kategoorias"
664,2025-04-13 09:46:43,18-29,Naine,Harju maakond,Eesti keel,"Jah, üle 6 jäätmekategoorias"
665,2025-04-15 13:31:30,50-64,Naine,Harju maakond,Eesti keel,"Jah, 3-5 jäätmekategoorias"
666,2025-03-11 19:48:03,30-49,Naine,Harju maakond,Eesti keel,"Jah, 3-5 jäätmekategoorias"



### Group D — Optional Purchasing Module (Q28–Q34)

**Key check first:** Q28 is the gateway — only respondents who answered "Jah" (564) should have data in Q29–Q34. The 114–115 NaNs in Q29–Q34 align with Q28="Ei" (111) + 3 NaN on Q28. **Verify this alignment row-by-row** — any "Ei" on Q28 with non-null Q29–Q34 is a data entry error.

| Col | Variable | Checks |
|---|---|---|
| **Q28** | Consent to purchasing questions | 3 NaNs — treat as missing consent; exclude from purchasing module |
| **Q29** | Wardrobe satisfaction (1–5 as float) | Stored as float — verify no values outside 1–5; check NaN count matches Q28="Ei" exactly |
| **Q30** | How long do you wear clothes | 15 unique values — check for expected ordinal categories; free-text in Q30.1 |
| **Q31** | Purchase frequency | 6 values — check expected ordinal set; NaN count should match Q28="Ei" |
| **Q32** | Ultra-fast fashion purchases (float) | Values 1–5 — check scale direction (1=never? 1=always?); NaNs should match Q28="Ei" |
| **Q33** | Reason for ultra-fast fashion | Conditional on Q32 ≠ "never" — 511 NaNs expected; verify non-null count is consistent with Q32 responses indicating "yes" |
| **Q34** | Online order-and-return frequency (float) | Values 1–5 — check scale codebook; NaNs (111) should align with Q28="Ei" |

---

### Group E — Open Text (flag, do not clean numerically)

Columns Q9.1, Q10.1, Q13.1, Q14.1, Q15.1, Q18.1, Q20.1, Q23.1, Q26, Q30.1 are optional elaborations. Checks: verify high missingness is expected (it is — these are "if you wish" fields); check that respondents who chose specific answer options in the parent question did provide elaboration where it was logically mandatory (e.g., Q20.1 when Q20 = "Olen kahelnud").

---

### Summary of Critical Actions Before Analysis

1. **Exclude 5 respondents** who answered "Ei" on Q2
2. **Resolve Q21 mixed encoding** (text labels + numeric codes 1–4 coexist)
3. **Parse all multi-select columns** (Q7, Q13, Q15, Q17, Q18, Q19, Q22, Q24, Q25) into binary dummy variables
4. **Segment Q12** into usable ordinal vs. free-text unusable
5. **Validate Q28 gateway logic** — NaN counts in Q29–Q34 must match Q28 non-consenting rows exactly
6. **Document Q5 location standardization** strategy (county mapping vs. urban/rural binary)

## Andmete eksploratiivne analüüs

### Q6_keel

In [297]:
# Leia vastusevariandid "muu" ning nende osas antud täpsustus
duckdb.sql('''
    SELECT
        Q6_keel_tekst
    FROM data_raw t1
    JOIN vastuste_koodid t2 ON t1.Q6_keel = t2.kood
        AND t2.kysimus = 'Q6_keel'
    WHERE t2.vastus = 'Muu'
''').df()

,Q6_keel_tekst
0,Eesti keel / vene keel
1,Jumanji
2,Eesti ja vene keel
3,eesti + hollandi + inglise
4,Valgevene keel
5,Hiina


### Q7_sorteerimiskaitumine

In [298]:
# Leia vastusevariandid "muu" ning nende osas antud täpsustus
duckdb.sql('''
    SELECT
        Q7_sorteerimiskaitumine_tekst
    FROM data_raw t1
    JOIN vastuste_koodid t2 ON t1.Q7_sorteerimiskaitumine = t2.kood
        AND t2.kysimus = 'Q7_sorteerimiskaitumine'
    WHERE t2.vastus = 'Muu'
''').df()

,Q7_sorteerimiskaitumine_tekst
0,Ei tea
1,"Tekstiilid, mida saan õmblemiseks, kaltsuvaiba..."
2,"Jah, kõik võimalikud sorteerimised"
3,ühistu mittetoimivuse tõttu on teema raskendatud
4,"Pole eri kaste, seega ei sorteeri"
5,Me põledame prügi ära


### Q9_probleemi_tosidus

In [294]:
# Milliseid täpsustusi lisati probleemi tõsiduse kirjeldamiseks?
duckdb.sql('''
    SELECT DISTINCT Q10_probleemi_tosidus_tapsustus
    FROM data_raw
''').df()

,Q10_probleemi_tosidus_tapsustus
0,Viimastel aastatel kasutatud asjade ost popula...
1,edasi teha nii: 1.
2,Ei saa küsimusest aru\n
3,Ületootmine. Rõivaid kasutatakse vähe. Kvalite...
4,Naftast riideesemete väga suur hulk. Kiirmood....
...,...
113,"Leian, et tegu on olulise probleemiga aga mitt..."
114,Inimeste teadlikkus ja tegutsemise soov on üsn...
115,"1. Kiired trendid, osad nimesed ostavad riidei..."
116,"Pole aimugi , mis teised teevad"


### Q11_teadlikkus

In [303]:
# Milliseid täpsustusi lisati teadlikkuse kirjeldamiseks?
duckdb.sql('''
    SELECT DISTINCT Q12_teadlikkus_tapsustus
    FROM data_raw
''').df()

,Q12_teadlikkus_tapsustus
0,-
1,Aga kahjuks KOV hiilivad sellest eemale.
2,Tagastan parandamatud riided rõiva kauplustess...
3,"Mõttetu Eestis, see kogumine palju suurema jal..."
4,Infot napib
...,...
48,Kuna tegemist on järjekordse mõttetu seadusega...
49,Väikestes maakohtades ei ole võimalik tekstiil...
50,Samas kuhu viia eraldi? Seda infot pole. Ka va...
51,"Ma ei teadnud, et segaolmesse ei tohi panna"
